# DTGS183. Figure 1c: Simulating Bacteria

This notebook simulates a brightfield image of a bacterial sample with DeepTrack2, reproducing the bacteria panel of Figure 1c. Two custom `VolumeScatterer` subclasses model rod-shaped bacilli and short chains of dividing bacilli, which are combined into a mixed population. The notebook produces two matching panels: the simulated brightfield image, and a ground-truth/label overlay distinguishing lone bacilli from dividing chains.


In [ ]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.


In [ ]:
import random

import numpy as np
from matplotlib import pyplot as plt
from skimage.segmentation import find_boundaries

import deeptrack as dt
from deeptrack import units as u
from deeptrack.backend.units import ConversionTable
from deeptrack.optical.scatterers import VolumeScatterer


def seed_everything(seed: int = 123) -> None:
    random.seed(seed)
    np.random.seed(seed)


SEED = 345
seed_everything(SEED)

TILE_SIZE = 256
RESOLUTION = 1e-7  # 0.656 um/px, physically corrected resolution.


## 1. Defining the Bacteria Shapes

`Bacillus` renders a single rod-shaped bacterium as a hollow shell: a capsule (cylinder with hemispherical caps) with wave-like bending along its length, minus a smaller inner capsule to leave only the cell envelope. `ChainBacillus` extends the same shell construction to a short chain of `n_bacilli` segments, hinged end-to-end along a bounded random walk in heading to model a dividing/elongating chain.


In [ ]:
class Bacillus(VolumeScatterer):
    __conversion_table__ = ConversionTable(
        radius=(u.meter, u.meter),
        length=(u.meter, u.meter),
        rotation=(u.radian, u.radian),
    )

    def __init__(
        self,
        radius=0.4e-6,
        length=2e-6,
        rotation=(0, 0, 0),
        bend=0,
        refractive_index=1.38,
        **kwargs,
    ):
        super().__init__(
            radius=radius,
            length=length,
            rotation=rotation,
            bend=bend,
            refractive_index=refractive_index,
            **kwargs,
        )

    def _process_properties(self, properties):
        properties = super()._process_properties(properties)

        # Ensure radius is scalar
        radius = np.array(properties["radius"])
        if radius.size > 1:
            radius = radius[0]
        properties["radius"] = float(radius)

        # Ensure length is scalar
        properties["length"] = float(properties["length"])

        # Ensure rotation is 3 values
        rotation = np.array(properties["rotation"])
        if rotation.ndim == 0:
            rotation = [rotation, 0, 0]
        elif rotation.size == 1:
            rotation = [rotation[0], 0, 0]
        elif rotation.size == 2:
            rotation = [rotation[0], rotation[1], 0]
        properties["rotation"] = tuple(rotation)

        return properties

    def get(
        self,
        *ignore,
        radius,
        length,
        rotation,
        bend,
        voxel_size,
        **kwargs,
    ):

        # Determine grid size
        rad_ceil = np.ceil(radius / np.min(voxel_size[:2]))
        len_ceil = np.ceil(length * 0.5 / voxel_size[2]) + rad_ceil
        ceil = int(max(rad_ceil, len_ceil))

        # Create the grid
        x = np.arange(-ceil, ceil) * voxel_size[0]
        y = np.arange(-ceil, ceil) * voxel_size[1]
        z = np.arange(-ceil, ceil) * voxel_size[2]
        Y, X, Z = np.meshgrid(y, x, z)

        # Rotate the grid
        cos = np.cos(rotation)
        sin = np.sin(rotation)
        XR = (
            cos[0] * cos[1] * X
            + (cos[0] * sin[1] * sin[2] - sin[0] * cos[2]) * Y
            + (cos[0] * sin[1] * cos[2] + sin[0] * sin[2]) * Z
        )
        YR = (
            sin[0] * cos[1] * X
            + (sin[0] * sin[1] * sin[2] + cos[0] * cos[2]) * Y
            + (sin[0] * sin[1] * cos[2] - cos[0] * sin[2]) * Z
        )
        ZR = (-sin[1] * X) + cos[1] * sin[2] * Y + cos[1] * cos[2] * Z

        # Apply wave-like bending along the length of the bacillus
        lateral_offset_x = bend * np.cos(np.pi * YR / length)
        XR -= lateral_offset_x

        # Cylinder mask
        cyl_mask = ((XR**2 / radius**2 + ZR**2 / radius**2) < 1) & (
            np.abs(YR) < (length / 2)
        )
        # Hemispherical caps
        cap_mask = (
            XR**2 + ZR**2 + (np.abs(YR) - (length / 2)) ** 2
        ) < radius**2

        # Combine cylinder with end-caps to form the full bacillus shape
        mask = cyl_mask | cap_mask

        # Hollow out an inner capsule to leave only the cell envelope (shell)
        inner_radius = radius * 0.6
        inner_length = length * 0.9

        cyl_mask_in = ((XR**2 / inner_radius**2 + ZR**2 / inner_radius**2) < 1) & (
            np.abs(YR) < (inner_length / 2)
        )
        cap_mask_in = (
            XR**2 + ZR**2 + (np.abs(YR) - (inner_length / 2)) ** 2
        ) < inner_radius**2
        mask_in = cyl_mask_in | cap_mask_in

        shell = mask & ~mask_in
        return shell.astype(float)


In [ ]:
class ChainBacillus(VolumeScatterer):
    __conversion_table__ = ConversionTable(
        radius=(u.meter, u.meter),
        length=(u.meter, u.meter),
        rotation=(u.radian, u.radian),
        chain_bend=(u.radian, u.radian),  # an ANGLE (per-joint)
    )

    def __init__(
        self,
        radius=0.4e-6,
        length=2e-6,
        n_bacilli=5,
        rotation=(0, 0, 0),
        chain_bend=0,
        bacillus_bend=0,
        overlap_fraction=0.02,   # fraction of capsule half-length consecutive segments overlap by
        **kwargs
    ):
        super().__init__(
            radius=radius,
            length=length,
            n_bacilli=n_bacilli,
            rotation=rotation,
            chain_bend=chain_bend,
            bacillus_bend=bacillus_bend,
            overlap_fraction=overlap_fraction,
            **kwargs
        )

    def _process_properties(self, properties):
        properties = super()._process_properties(properties)

        radius = np.array(properties["radius"])
        if radius.size > 1:
            radius = radius[0]
        properties["radius"] = float(radius)

        length = properties["length"]
        if np.ndim(length) == 0:
            properties["length"] = float(length)
        else:
            properties["length"] = np.array(length, dtype=float)

        properties["n_bacilli"] = int(properties.get("n_bacilli"))

        rotation = np.array(properties["rotation"])
        if rotation.ndim == 0:
            rotation = [rotation, 0, 0]
        elif rotation.size == 1:
            rotation = [rotation[0], 0, 0]
        elif rotation.size == 2:
            rotation = [rotation[0], rotation[1], 0]
        properties["rotation"] = tuple(rotation)

        return properties

    def get(
        self,
        *ignore,
        radius,
        length,
        n_bacilli,
        rotation,
        chain_bend,
        bacillus_bend,
        overlap_fraction,
        voxel_size,
        **kwargs
    ):
        # Determine grid size
        rad_ceil = np.ceil(radius / np.min(voxel_size[:2]))
        len_ceil = (
            np.ceil(length * 0.5 * n_bacilli / voxel_size[2])
            + rad_ceil * n_bacilli
        )
        ceil = int(max(rad_ceil, len_ceil))

        # Create grid
        x = np.arange(-ceil, ceil) * voxel_size[0]
        y = np.arange(-ceil, ceil) * voxel_size[1]
        z = np.arange(-ceil, ceil) * voxel_size[2]
        Y, X, Z = np.meshgrid(y, x, z)

        # Base 3D rotation of the whole chain 
        cos = np.cos(rotation)
        sin = np.sin(rotation)
        XR = (
            cos[0] * cos[1] * X
            + (cos[0] * sin[1] * sin[2] - sin[0] * cos[2]) * Y
            + (cos[0] * sin[1] * cos[2] + sin[0] * sin[2]) * Z
        )
        YR = (
            sin[0] * cos[1] * X
            + (sin[0] * sin[1] * sin[2] + cos[0] * cos[2]) * Y
            + (sin[0] * sin[1] * cos[2] - cos[0] * sin[2]) * Z
        )
        ZR = (-sin[1] * X) + cos[1] * sin[2] * Y + cos[1] * cos[2] * Z

        # Consecutive segments are spaced slightly CLOSER than exact
        # end-to-end tangency
        capsule_half_length = length / 2 + radius
        place_half_length = capsule_half_length * (1 - overlap_fraction)

        # True hinge construction
        heading = 0.0
        headings = [heading]
        hinge = np.array([0.0, 0.0])
        direction = np.array([np.sin(heading), np.cos(heading)])
        center = hinge + place_half_length * direction
        centers = [center]
        hinge = center + place_half_length * direction  # this segment's far end

        for i in range(1, n_bacilli):
            heading += chain_bend
            headings.append(heading)
            direction = np.array([np.sin(heading), np.cos(heading)])
            center = hinge + place_half_length * direction
            centers.append(center)
            hinge = center + place_half_length * direction  # next joint

        centers = np.array(centers)
        mean_xy = centers.mean(axis=0)

        shell = np.zeros_like(XR, dtype=bool)
        for (cx, cy), seg_heading in zip(centers, headings):
            dx = XR - (cx - mean_xy[0])
            dy = YR - (cy - mean_xy[1])
            local_x = dx*np.cos(seg_heading) - dy*np.sin(seg_heading)
            local_y = dx*np.sin(seg_heading) + dy*np.cos(seg_heading)
            local_z = ZR

            lateral_offset_x = bacillus_bend * np.cos(np.pi * local_y / length)
            local_x = local_x - lateral_offset_x

            cyl_mask = (
                (local_x**2 / radius**2 + local_z**2 / radius**2) < 1
            ) & (np.abs(local_y) < length / 2)

            cap_mask = (
                local_x**2 + local_z**2 + (np.abs(local_y) - length / 2) ** 2
            ) < radius**2

            mask0 = cyl_mask | cap_mask

            inner_radius = radius * 0.6
            inner_length = length * 0.9

            cyl_mask_in = ((local_x**2 / inner_radius**2 + local_z**2 / inner_radius**2) < 1) & (
                np.abs(local_y) < (inner_length / 2)
            )
            cap_mask_in = (
                local_x**2 + local_z**2 + (np.abs(local_y) - (inner_length / 2)) ** 2
            ) < inner_radius**2
            mask_in = cyl_mask_in | cap_mask_in

            shell |= mask0 & ~mask_in

        return shell.astype(float)


## 2. Setting Up the Optics

Brightfield, darkfield, and fluorescence optics are configured over the same `TILE_SIZE` output region, so that all three imaging modes can later be resolved from the same sample.


In [ ]:
optics_bf = dt.Brightfield(resolution=RESOLUTION, magnification=1,
    NA=1.2, refractive_index_medium=1.33,
    output_region=(0, 0, TILE_SIZE, TILE_SIZE)
    )

optics_df = dt.Darkfield(resolution=RESOLUTION, magnification=1,
    NA=1.45, refractive_index_medium=1.33,
    output_region=(0, 0, TILE_SIZE, TILE_SIZE)
    )

optics_fluo = dt.Fluorescence(resolution=RESOLUTION, magnification=1,
    NA=1.0,
    output_region=(0, 0, TILE_SIZE, TILE_SIZE)
    )


## 3. Simulating the Sample

A population of lone bacilli (`multiple_bacilli`) and a population of short dividing chains (`multiple_chains`) are combined into one sample, `bac`. Each population is also passed through `dt.SampleToMasks` to build binary masks (`mask_lone`, `mask_dividing`) labeling which pixels belong to each class, matched to the brightfield, darkfield, and fluorescence images by resolving all five features together in a single pipeline.


In [ ]:
max_bacillus_bend = 2e-7
bacillus = Bacillus(
    position=lambda: np.random.uniform(10, TILE_SIZE-10, 2),
    rotation=lambda: np.random.uniform(0, 2 * np.pi),
    length=lambda: np.random.uniform(1.4e-6, 2.2e-6),
    bend=lambda: (np.random.rand() - 0.5) * 2 * max_bacillus_bend,
    refractive_index=lambda: np.random.uniform(1.38, 1.42),
)

max_bacillus_bend2 = 1e-7
max_chain_bend = np.pi / 10

chain_bacilli = ChainBacillus(
    position=lambda: np.random.uniform(10, TILE_SIZE-10, 2),
    length=lambda: np.random.uniform(1.4e-6, 1.8e-6),
    n_bacilli=2,
    rotation=lambda: np.random.uniform(0, 2 * np.pi),
    chain_bend=lambda: (np.random.rand() - 0.5) * 2 * (max_chain_bend),
    bacillus_bend=lambda: (np.random.rand() - 0.5) * 2 * max_bacillus_bend2,
    refractive_index=lambda: np.random.uniform(1.38, 1.42),
)

multiple_bacilli = bacillus ^ (lambda: np.random.randint(5, 10))
multiple_chains = chain_bacilli ^ (lambda: np.random.randint(3, 4))

bac = multiple_bacilli & multiple_chains


def get_mask(radius):
    """Apply isotropic erosion to a binary mask."""
    def inner(mask):
        mask = np.sum(mask, -1, keepdims=True) > 0
        return mask
    return inner

mask_lone = (
    multiple_bacilli
    >> dt.SampleToMasks(get_mask, radius=0, output_region=(0, 0, TILE_SIZE, TILE_SIZE),
                       merge_method="or")
)

mask_dividing = (
    multiple_chains
    >> dt.SampleToMasks(get_mask, radius=0, output_region=(0, 0, TILE_SIZE, TILE_SIZE),
                       merge_method="or")
)


In [ ]:
pip_bf = optics_bf(bac) >> dt.Gaussian(sigma=0.035) >> dt.GaussianBlur(sigma=2.4)
pip_df = optics_df(bac) >> dt.Gaussian(sigma=0.00135) >> dt.GaussianBlur(sigma=1.6)
pip_fluo = optics_fluo(bac) >> dt.Poisson(snr=10)
pip = pip_bf & pip_df & pip_fluo & mask_lone & mask_dividing

pip.update()
image_bf, image_df, image_fluo, mask_lone, mask_dividing = pip.resolve()


## 4. Plotting the Brightfield Panel

The brightfield image is shown with a physical scale bar and saved as the main panel. Darkfield and fluorescence renders of the same sample are also previewed here for comparison, but not saved.

In [ ]:
px_per_um = 1.0 / (RESOLUTION * 1e6)   # RESOLUTION is in meters/px
SCALE_BAR_UM = 5
scale_bar_px = SCALE_BAR_UM * px_per_um

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(image_bf, cmap="gray")
x0, y0 = TILE_SIZE * 0.06, TILE_SIZE * 0.94
ax.plot([x0, x0 + scale_bar_px], [y0, y0], color="white", linewidth=2.5,
        solid_capstyle="butt")
ax.axis("off")
plt.show()
fig.savefig("fig1c_bacteria_sim.svg", format="svg", bbox_inches="tight", pad_inches=0, transparent=True)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(image_df, cmap="gray")
x0, y0 = TILE_SIZE * 0.06, TILE_SIZE * 0.94
ax.plot([x0, x0 + scale_bar_px], [y0, y0], color="white", linewidth=2.5,
        solid_capstyle="butt")
ax.axis("off")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(image_fluo, cmap="gray")
x0, y0 = TILE_SIZE * 0.06, TILE_SIZE * 0.94
ax.plot([x0, x0 + scale_bar_px], [y0, y0], color="white", linewidth=2.5,
        solid_capstyle="butt")
ax.axis("off")
plt.show()


## 5. Plotting the Ground-Truth Mask

The `mask_lone`/`mask_dividing` masks resolved above are combined into a single color-coded ground-truth panel: lone bacilli and dividing chains are colored differently and outlined, then saved alongside the brightfield panel.

In [ ]:
COLOR_LONE = np.array([0, 114/255, 178/255])       # Wong blue
COLOR_DIVIDING = np.array([213/255, 94/255, 0])    # Wong vermillion

lone_bool = mask_lone.squeeze().astype(bool)   # isolated bacteria mask
dividing_bool = mask_dividing.squeeze().astype(bool)  # dividing bacteria mask

# --- Composite class map ---
composite_gt = np.ones((mask_lone.shape[0], mask_lone.shape[1], 3), dtype=np.float32)
composite_gt[lone_bool] = COLOR_LONE
composite_gt[dividing_bool] = COLOR_DIVIDING

overlap = lone_bool & dividing_bool
if overlap.any():
    composite_gt[overlap] = [0.4, 0.4, 0.4]  # flag any unexpected overlap

boundaries = find_boundaries(lone_bool | dividing_bool, mode="inner")
composite_gt[boundaries] = [0, 0, 0]

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(composite_gt)
ax.axis("off")
plt.show()
fig.savefig("fig1c_bacteria_gt.svg", format="svg", bbox_inches="tight", pad_inches=0, transparent=True)
